# SepsisCare 完整安装向导

版本：0.9.1  
适用交付目录：`SepsisCare_Final_Delivery_20260518`  
目标：完成软件安装包、模型部署包、DeepSeek 配置、训练终端连接和基础 smoke test。

> 注意：本向导不包含任何真实 API Key。DeepSeek Key 需要在安装后的软件设置页手动填写，或通过后端运行时配置写入。不要把 key 写进源码、安装包、论文或截图。

## 1. 最终交付物结构

最终交付目录应包含以下内容：

```text
SepsisCare_Final_Delivery_20260518/
  01_software_installers/
    sepsiscare-0.9.0-macOS.dmg
    sepsiscare-0.9.0-win-x64.exe
    sepsiscare-0.9.0-win-arm64.exe
    sepsiscare-0.9.1-release.apk
  02_model_deploy_package/
    deploy/start_model_service.sh
    deploy/model_service.py
    models/cloud_production/s7_phenotype_contrastive_full_20260516/
    config/s7_phenotype_contrastive_full_20260516.yaml
  03_documentation/
    SepsisCare_Paper_Report.tex
    SepsisCare_Installation_Guide.ipynb
  SepsisCare_software_installers_20260518.zip
  SepsisCare_model_deploy_package_20260518.zip
  SHA256SUMS.txt
```

三类交付物的职责：

| 交付物 | 内容 | 用途 |
|---|---|---|
| 软件安装包 | macOS DMG、Windows EXE、Android APK | 新电脑直接安装软件 |
| 模型本体 | S7 模型、读出器、云端 model service | 云端算力服务器部署与训练终端连接 |
| 全套文档 | 安装向导、论文报告、训练终端说明、部署说明 | 答辩、验收、复现与维护 |

## 2. 系统连接图

```mermaid
flowchart LR
  A[macOS SwiftUI App] --> API[HTTP JSON API]
  B[Windows Electron App] --> API
  C[Android WebView App] --> API
  API --> S[server.py local/cloud backend]
  S --> D[runtime_data demo/history]
  S --> L[DeepSeek Chat API]
  S --> T[training_terminal.py]
  T --> M[cloud model_service.py]
  M --> W[S7 model weights/readout/logs]
```

安装包默认支持本地演示；正式生产模型训练和大规模推理应部署到云端算力服务器。

## 3. 安装前检查

安装前确认：

1. 已拿到最终交付目录或两个 zip：
   - `SepsisCare_software_installers_20260518.zip`
   - `SepsisCare_model_deploy_package_20260518.zip`
2. macOS/Windows 端用于课程演示可直接安装。
3. Android 端是客户端，需要连接同局域网电脑后端或云端 API。
4. DeepSeek 智能体需要用户自行配置有效 API Key。
5. 云端真实训练需要服务器、Python 环境、模型部署包、足够存储和完整训练数据。

In [ ]:
# 可选：在 macOS/Linux 上校验最终交付包哈希。
# 修改 DELIVERY_ROOT 为实际交付目录路径。
DELIVERY_ROOT = "/Users/exusiaihy/Desktop/SepsisCare_Final_Delivery_20260518"
print(DELIVERY_ROOT)

In [ ]:
# macOS/Linux 可执行：
# !cd "$DELIVERY_ROOT" && shasum -c SHA256SUMS.txt

# Windows PowerShell 可参考：
# Get-FileHash .\SepsisCare_software_installers_20260518.zip -Algorithm SHA256

## 4. macOS 安装步骤

安装文件：

```text
01_software_installers/sepsiscare-0.9.0-macOS.dmg
```

步骤：

1. 双击 `.dmg`。
2. 将 `SepsisCare-macOS.app` 拖入 `Applications`。
3. 第一次打开如果提示安全限制，进入 `系统设置 > 隐私与安全性`，选择允许打开。
4. 启动 App 后，软件会自动尝试启动本地 `server.py` 后端。
5. 在软件内查看后端状态，显示 `本地后端在线` 即安装成功。

macOS 本地后端默认地址：

```text
http://127.0.0.1:8765
```

如果后端未自动启动，可在软件的后端状态页点击重启，或手动运行：

```bash
python3 /Applications/SepsisCare-macOS.app/Contents/Resources/backend/server.py --host 127.0.0.1 --port 8765
```

## 5. Windows 安装步骤

安装文件二选一：

```text
01_software_installers/sepsiscare-0.9.0-win-x64.exe
01_software_installers/sepsiscare-0.9.0-win-arm64.exe
```

选择规则：

| 电脑类型 | 安装包 |
|---|---|
| 常见 Intel/AMD Windows 笔记本或台式机 | `win-x64.exe` |
| Windows ARM 设备 | `win-arm64.exe` |

步骤：

1. 双击 EXE 安装包。
2. 按安装向导选择安装目录。
3. 允许创建桌面快捷方式和开始菜单快捷方式。
4. 如系统提示防火墙，允许 SepsisCare 使用本地网络。
5. 启动软件后进入管理员端或服务监控页，确认 API 在线。

静默安装可选命令：

```powershell
.\sepsiscare-0.9.0-win-x64.exe /S
```

## 6. Android 安装步骤

安装文件：

```text
01_software_installers/sepsiscare-0.9.1-release.apk
```

步骤：

1. 将 APK 复制到 Android 手机。
2. 打开 APK，允许来自当前来源的安装。
3. 安装后打开 SepsisCare。
4. Android 不内置 Python 后端，必须连接一个可访问 API：
   - 同局域网 macOS/Windows 后端，例如 `http://电脑局域网IP:8765`
   - 云端服务器 API，例如 `http://服务器IP:端口`

电脑端如需让手机访问本地后端，应以局域网方式启动：

```bash
python3 server.py --host 0.0.0.0 --port 8765
```

然后在手机端配置 API 地址：

```text
http://电脑局域网IP:8765
```

## 7. DeepSeek 智能体配置

DeepSeek Key 不会也不应该内置在安装包中。新电脑安装后必须配置一次。

macOS：

```text
设置 > DeepSeek > 输入 API Key > 保存配置
```

Windows / Android / iOS Web 端：

```text
管理员端 > 服务监控 > DeepSeek 配置 > 输入 API Key > 保存配置
```

推荐配置：

| 字段 | 推荐值 |
|---|---|
| Provider | DeepSeek |
| Model | `deepseek-chat` |
| Base URL | `https://api.deepseek.com/chat/completions` |
| Timeout | `18` 秒 |

配置后，家属端智能体沟通接口 `/api/family/chat` 应返回：

```json
{
  "source": "deepseek",
  "model": "deepseek-chat",
  "configured": true
}
```

如果返回 `local_demo_fallback`，说明 DeepSeek 网络、Key 或后端配置仍有问题。

## 8. 本地后端 Smoke Test

后端启动后，在终端测试以下接口。

In [ ]:
# 健康检查
# !curl -s http://127.0.0.1:8765/health

# AI 配置状态
# !curl -s http://127.0.0.1:8765/api/config/ai

# 家属端 DeepSeek 测试。请先在软件里配置 DeepSeek Key。
# !curl -s -X POST http://127.0.0.1:8765/api/family/chat \
#   -H 'Content-Type: application/json' \
#   -d '{"question":"请用一句话说明当前状态","patient_ref":"P202605-001"}'

## 9. 云端模型部署包安装

模型部署包用于云端算力服务器或本地模拟云端服务。

部署目录：

```text
02_model_deploy_package/
```

一键启动：

```bash
cd 02_model_deploy_package
bash deploy/start_model_service.sh
```

默认服务地址：

```text
http://SERVER_IP:8788
```

模型服务接口：

| 接口 | 方法 | 用途 |
|---|---|---|
| `/health` | GET | 服务健康检查 |
| `/api/model/status` | GET | 当前模型与训练状态 |
| `/api/training/command` | POST | 接收训练终端动作或命令 |
| `/api/artifacts/latest` | GET | 下载/查看最新模型成果 |

默认是 protected deployment-demo 模式：可以回显状态、日志和指标，但不会自动启动昂贵训练。真实训练前需要完整数据和 GPU 环境。

In [ ]:
# 云端模型服务 smoke test。将 SERVER_IP 改成服务器 IP。
SERVER_IP = "127.0.0.1"
MODEL_PORT = 8788
print(f"http://{SERVER_IP}:{MODEL_PORT}/health")
# !curl -s http://127.0.0.1:8788/health
# !curl -s http://127.0.0.1:8788/api/model/status

## 10. 训练终端连接云端模型

在 SepsisCare 软件内打开：

```text
模型训练终端
```

配置：

| 字段 | 示例 |
|---|---|
| 模式 | `production` |
| 云端训练服务地址 | `http://SERVER_IP:8788` |
| 模型版本 | `S7-contrastive-20260516` |
| 数据版本 | `packaged-history-20260518` 或生产数据版本 |

快捷按钮：

| 快捷键 | 功能 |
|---|---|
| Ctrl/Command + 1 | 更新业务数据库 |
| Ctrl/Command + 2 | 继续训练 |
| Ctrl/Command + 3 | 暂停训练 |
| Ctrl/Command + 4 | 下载权重、数据集、日志 |
| Ctrl/Command + 5 | 查看实时日志、loss、accuracy、F1 |
| Ctrl/Command + 6 | 切换本地演示/云端生产模型 |
| Ctrl/Command + 7 | 重置训练参数 |
| Ctrl/Command + 8 | 同步本地配置至云端 |

生产模式下，训练终端会把动作转发到：

```text
http://SERVER_IP:8788/api/training/command
```

## 11. 常见故障排查

| 问题 | 可能原因 | 处理方法 |
|---|---|---|
| 软件打开后 API 没连上 | 后端未启动或端口被占用 | 重启 App，或手动运行 `server.py --port 8765` |
| 家属端不能连 DeepSeek | 未配置 Key、Key 无效、网络失败 | 进入 DeepSeek 配置页保存 Key，再测试 `/api/family/chat` |
| Android 无数据 | 手机不能访问电脑后端 | 后端用 `--host 0.0.0.0`，手机填电脑局域网 IP |
| 训练终端生产模式失败 | 云端地址未填或服务未启动 | 启动 `deploy/start_model_service.sh`，检查 `SERVER_IP:8788/health` |
| 模型不能真实继续训练 | 默认是 demo protected 模式 | GPU 服务器设置 `SEPSISCARE_ALLOW_REAL_TRAINING=1` 并准备完整数据 |
| 新电脑看不到旧历史 | 旧历史只在原电脑本地 | 需要导出/导入历史数据或把历史打包进 runtime_data |

重要边界：

1. 新电脑安装后可以使用安装包内置演示数据。
2. 新电脑不能自动看到旧电脑本地历史数据，除非手动迁移。
3. 新电脑不能直接跑真实大模型训练，除非安装模型部署包并连接有数据和算力的服务器。
4. DeepSeek Key 每台新电脑都需要重新配置。

## 12. 卸载与重装

macOS：

1. 删除 `/Applications/SepsisCare-macOS.app`。
2. 如需清除本机运行时配置，删除：

```text
~/Library/Application Support/SepsisCare/
~/.sepsiscare/
```

Windows：

1. 通过系统设置或控制面板卸载 SepsisCare。
2. 如需清除运行时配置，删除 `%APPDATA%\SepsisCare`。

Android：

1. 长按 App 图标卸载。
2. 重新安装 APK 后重新配置 API 地址。

## 13. 答辩演示建议流程

推荐演示顺序：

1. 展示最终交付目录：软件安装包、模型部署包、文档。
2. 打开 macOS 或 Windows 软件，确认后端在线。
3. 进入风险看板，展示患者列表、趋势图和历史 ICU 数据库。
4. 进入家属端，提问并展示 DeepSeek 返回的 `source=deepseek`。
5. 进入训练终端，展示 demo 模式快捷按钮和 production 模式云端地址配置。
6. 启动模型部署包的 `model_service.py`，展示 `/health` 和 `/api/model/status`。
7. 最后展示论文报告中的 S0-S7 pipeline 和工程架构图。